# 🛡️ Master Dataset Splitter & Balancer

Splits trimmed audio dataset into clean `train/`, `val/`, and `test/` sets:
1. **Validation Set (`val/`)**: **STRICT 1:1 RATIO** (50% gunshot, 50% non-gunshot).
2. **Test Set (`test/`)**: **STRICT 1:1 RATIO** (50% gunshot, 50% non-gunshot).
3. **Training Set (`train/`)**: Configurable ratio from **1:1 up to 1:30** (gunshot : non-gunshot).
4. **Zero Data Leakage**: **GroupKFold** by original source recording.

In [ ]:
# ============================================================
# CELL 1: Imports & Path Auto-Detection
# ============================================================
import os
import sys
import re
import json
import shutil
import random
from pathlib import Path
import pandas as pd
import numpy as np

# Auto-detect Project Root & Data Directory
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'Data'
if not DATA_DIR.exists() or len(PROJECT_ROOT.parts) <= 2:
    candidates = [
        Path(r'D:\Desktop\GunShot\Data-Cleaner\Data'),
        Path(r'D:\Desktop\GShot\Data-Cleaner\Data'),
        Path.cwd().resolve().parent / 'Data-Cleaner' / 'Data',
        Path.cwd().resolve().parent.parent / 'Data-Cleaner' / 'Data'
    ]
    for c in candidates:
        if c.exists():
            DATA_DIR = c
            PROJECT_ROOT = c.parent
            break

print(f'Project Root : {PROJECT_ROOT}')
print(f'Data Directory: {DATA_DIR}')

In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================
# Target clip duration in ms (250, 500, 750, 1000)
TARGET_MS = 750

# Train ratio: 1 gunshot : N non-gunshots (1 to 30)
# Recommended for training: 5 (1:5 ratio)
TRAIN_RATIO = 5

# GroupKFold split fractions
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

OUTPUT_DIR = DATA_DIR / f'SPLIT_DATASET_{TARGET_MS}MS'

print(f'Target Duration   : {TARGET_MS}ms')
print(f'Train Ratio       : 1:{TRAIN_RATIO} (Gunshot : Non-Gunshot)')
print(f'Validation Ratio  : 1:1 STRICT')
print(f'Test Ratio        : 1:1 STRICT')
print(f'Output Directory  : {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 3: Splitter Engine Functions
# ============================================================

def extract_source_group(filepath):
    stem = Path(filepath).stem
    no_prefix = re.sub(r'^(c[01]_|rej_)\d+_', '', stem)
    no_suffix = re.sub(r'_(onset\d+|win\d+|clip\d+).*$', '', no_prefix)
    no_aug = re.sub(r'_aug_\d+dB$', '', no_suffix)
    return no_aug if no_aug else stem

def balance_1to1(file_list):
    class1 = [f for f, l in file_list if l == 1]
    class0 = [f for f, l in file_list if l == 0]
    min_len = min(len(class1), len(class0))
    if min_len == 0: return []
    random.shuffle(class1)
    random.shuffle(class0)
    res = [(f, 1) for f in class1[:min_len]] + [(f, 0) for f in class0[:min_len]]
    random.shuffle(res)
    return res

def sample_train_ratio(file_list, ratio=5):
    ratio = max(1, min(30, ratio))
    class1 = [f for f, l in file_list if l == 1]
    class0 = [f for f, l in file_list if l == 0]
    target0 = min(len(class0), len(class1) * ratio)
    random.shuffle(class1)
    random.shuffle(class0)
    res = [(f, 1) for f in class1] + [(f, 0) for f in class0[:target0]]
    random.shuffle(res)
    return res

print('✅ Splitter engine functions defined.')

In [ ]:
# ============================================================
# CELL 4: EXECUTE DATASET SPLITTING
# ============================================================
random.seed(42)
np.random.seed(42)

# Locate Audio Folders
gunshot_dir = DATA_DIR / f'TRIMMED_GUNSHOTS_{TARGET_MS}MS' / 'verified'
if not gunshot_dir.exists(): gunshot_dir = DATA_DIR / 'TRIMMED_GUNSHOTS' / 'verified'
if not gunshot_dir.exists(): gunshot_dir = DATA_DIR / 'gun'

nongunshot_dir = DATA_DIR / f'TRIMMED_NONGUNSHOTS_{TARGET_MS}MS' / 'clean'
if not nongunshot_dir.exists(): nongunshot_dir = DATA_DIR / 'TRIMMED_NONGUNSHOTS' / 'clean'
if not nongunshot_dir.exists(): nongunshot_dir = DATA_DIR / 'sound'

gunshot_files = sorted(list(gunshot_dir.rglob('*.wav')))
nongunshot_files = sorted(list(nongunshot_dir.rglob('*.wav')))

print(f'Gunshot Source Directory    : {gunshot_dir.name} ({len(gunshot_files):,} clips)')
print(f'Non-Gunshot Source Directory: {nongunshot_dir.name} ({len(nongunshot_files):,} clips)\n')

# Grouping by Source File (GroupKFold)
groups = {}
for f in gunshot_files:
    g = extract_source_group(f)
    groups.setdefault(g, []).append((f, 1))
for f in nongunshot_files:
    g = extract_source_group(f)
    groups.setdefault(g, []).append((f, 0))

unique_groups = list(groups.keys())
random.shuffle(unique_groups)

n_test = max(1, int(len(unique_groups) * TEST_SPLIT))
n_val = max(1, int(len(unique_groups) * VAL_SPLIT))

test_g = set(unique_groups[:n_test])
val_g = set(unique_groups[n_test:n_test + n_val])
train_g = set(unique_groups[n_test + n_val:])

raw_train, raw_val, raw_test = [], [], []
for g, items in groups.items():
    if g in test_g:
        raw_test.extend(items)
    elif g in val_g:
        raw_val.extend(items)
    else:
        raw_train.extend(items)

final_val = balance_1to1(raw_val)
final_test = balance_1to1(raw_test)
final_train = sample_train_ratio(raw_train, ratio=TRAIN_RATIO)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

manifest_rows = []
print('Building output directory structure...')

for split_name, tuples in [('train', final_train), ('val', final_val), ('test', final_test)]:
    c1_dir = OUTPUT_DIR / split_name / 'class_1_gunshot'
    c0_dir = OUTPUT_DIR / split_name / 'class_0_nongunshot'
    c1_dir.mkdir(parents=True, exist_ok=True)
    c0_dir.mkdir(parents=True, exist_ok=True)
    
    c1_cnt, c0_cnt = 0, 0
    for src_path, label in tuples:
        dest_dir = c1_dir if label == 1 else c0_dir
        shutil.copy2(src_path, dest_dir / src_path.name)
        if label == 1: c1_cnt += 1
        else: c0_cnt += 1
        manifest_rows.append({
            'filename': src_path.name,
            'split': split_name,
            'label': label,
            'source_group': extract_source_group(src_path),
            'source_path': str(src_path)
        })
    
    ratio_str = f'1:{c0_cnt/max(1,c1_cnt):.1f}'
    print(f'  [{split_name.upper():5s}] Gunshots: {c1_cnt:>5,} | Non-Gunshots: {c0_cnt:>5,} | Ratio: {ratio_str}')

reports_dir = OUTPUT_DIR / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(manifest_rows).to_csv(reports_dir / 'split_manifest.csv', index=False)
print(f'\n✅ SPLIT COMPLETE! Dataset ready at: {OUTPUT_DIR}')